In [ ]:
import pandas as pd
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.drop(columns= 'Unnamed: 0.1', inplace= True)

# we convert 0-12 to radians circle for a seasonal mapping (circular time encoding)
# ensure index is treated as datetime (works even if the Index is not recognized as DatetimeIndex)
import numpy as np
month = pd.DatetimeIndex(df.index).month

df['year_sin'] = np.sin(2*np.pi*month/12)
df['year_cos'] = np.cos(2*np.pi*month/12)


df1 = df.copy()
df.head()


In [ ]:
import sktime

In [ ]:
from sktime.utils.plotting import plot_series

y = df1['runoff']

# plotting for visualization
plot_series(y)

In [ ]:
y.index

In [ ]:
y = df1['runoff']   # target
X = df1[['sca', 'dd', 'precipitation', 'et_loss']]  # raw features only

In [ ]:
y.index

In [ ]:
from sktime.forecasting.model_selection import temporal_train_test_split

y_train, y_test, X_train, X_test = temporal_train_test_split(
    y, X, test_size = 0.2
)

In [ ]:
from sktime.forecasting.arima import AutoARIMA

forecaster = AutoARIMA(
    seasonal=True,
    m=12,   # monthly data
    suppress_warnings=True
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

In [ ]:
from sktime.forecasting.ets import AutoETS

forecaster = AutoETS(
    seasonal='add',
    sp=12
)

In [ ]:
forecaster.fit(y_train)

In [ ]:
fh = list(range(1, len(y_test)+1))

y_pred = forecaster.predict(fh=fh, X=X_test)

y_pred = forecaster.predict(fh=fh, X=X_test)
print(fh)

In [ ]:
from utils import calculate_nse
print(calculate_nse(y_test.values, y_pred.values))

In [ ]:
import numpy as np

def nse(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)

print(nse(y_test.values, y_pred.values))

In [ ]:
import warnings

In [ ]:
from sktime.forecasting.model_selection import ExpandingWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import make_forecasting_scorer
# nse_scorer = make_forecasting_scorer(nse)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
 
cv = ExpandingWindowSplitter(
    initial_window=int(len(y)*0.6),
    step_length=12,
    fh = [1,2,3,4,5,6]
)
nse_scorer = make_forecasting_scorer(
    nse,
    name="nse",
    greater_is_better=True
)
results = evaluate(
    forecaster,
    y=y,
    X=X,
    cv=cv,
    strategy="refit",
    scoring=nse_scorer,
    error_score='raise'
)
# print(results)
print(results['test_nse'].mean())
# print(results.columns)
# print(results['test_nse'])
print(results[['test_nse', 'len_train_window', 'cutoff']])
print(results['test_nse'].std())
print(results['test_nse'].median())